In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from huggingface_hub import login
login()

In [3]:
!pip -q install -U transformers

import os, json, time, random, shutil, warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt, yaml
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm
import torch
from transformers import AutoModel
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = "/content/drive/MyDrive/lane detection"
DATA_DIR = f"{BASE_DIR}/data"
PROCESSED_DIR = f"{DATA_DIR}/processed"
LOCAL_PROCESSED = "/content/processed"
RESULTS_DIR = f"{BASE_DIR}/results"
PREDICTIONS_DIR = f"{BASE_DIR}/predictions"
CACHE_DIR = f"{BASE_DIR}/cache_raw"
FUSION_DIR = f"{BASE_DIR}/fusion_inputs"
STATS_DIR = f"{RESULTS_DIR}/test_stats"
RESULTS_FUSION_CSV = f"{RESULTS_DIR}/results_fusion.csv"
for d in [LOCAL_PROCESSED, FUSION_DIR, STATS_DIR]:
	os.makedirs(d, exist_ok=True)

W, H = 640, 224
DS = {'culane': {'w': 1640, 'h': 590, 'crop': 236}, 'bdd': {'w': 1280, 'h': 720, 'crop': 302}}
LINE_WIDTH = {'culane': 30, 'bdd': 23}
SPLIT_DS = {'val': 'culane', 'test_culane': 'culane', 'test_bdd': 'bdd'}
SPLITS = ['val', 'test_culane', 'test_bdd']
N_ROWS, N_CELLS = 36, 120
ROWS = np.linspace(10, H - 1, N_ROWS).astype(int)
X_MIN, X_MAX = -0.25 * W, 1.25 * W
CELL = (X_MAX - X_MIN) / N_CELLS
MIN_ROWS = 4
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)
SMOOTH = json.load(open(f"{RESULTS_DIR}/best_settings.json"))['smooth']
DINO = 'facebook/dinov3-vits16-pretrain-lvd1689m'

GT = {}
for split in SPLITS:
	GT[split] = (np.load(f"{PROCESSED_DIR}/{split}/lanes.npy"), pd.read_csv(f"{PROCESSED_DIR}/{split}/meta.csv"))
print('smoothing:', SMOOTH, '| GPU:', torch.cuda.is_available())

smoothing: 3 | GPU: True


In [4]:
def load(name):
	src = f"{PROCESSED_DIR}/{name}"
	dst = f"{LOCAL_PROCESSED}/{name}"
	complete = os.path.exists(f"{dst}/meta.csv") and os.path.exists(f"{dst}/images.npy")
	if complete:
		complete = os.path.getsize(f"{dst}/images.npy") == os.path.getsize(f"{src}/images.npy")
	if not complete:
		shutil.copytree(src, dst, dirs_exist_ok=True)
	return np.load(f"{dst}/images.npy", mmap_mode='r')

def resample(x, y):
	order = np.argsort(y)
	x = x[order]
	y = y[order]
	rows = np.arange(H)
	out = np.full(H, np.nan, np.float32)
	inside = (rows >= y[0]) & (rows <= y[-1])
	out[inside] = np.interp(rows[inside], y, x)
	return out

def to_orig(lanes, ds):
	c = DS[ds]
	x = lanes * c['w'] / W
	y = np.arange(H) * (c['h'] - c['crop']) / H + c['crop']
	return x, y

def lane_mask(x, ds):
	c = DS[ds]
	xo, yo = to_orig(x, ds)
	mask = np.zeros((c['h'] - c['crop'], c['w']), np.uint8)
	rows = np.where(np.isfinite(xo))[0]
	if len(rows) > 1:
		points = np.stack([xo[rows], yo[rows] - c['crop']], 1).round().astype(np.int32)
		cv2.polylines(mask, [points], False, 1, LINE_WIDTH[ds])
	return mask

def iou(a, b):
	union = np.count_nonzero(a | b)
	if union == 0:
		return 0.0
	return np.count_nonzero(a & b) / union

def frame_ious(pred, gt, ds):
	pred_masks = []
	gt_masks = []
	for k in range(2):
		if np.isfinite(pred[k]).sum() > 1:
			pred_masks.append(lane_mask(pred[k], ds))
		else:
			pred_masks.append(None)
		if np.isfinite(gt[k]).sum() > 1:
			gt_masks.append(lane_mask(gt[k], ds))
		else:
			gt_masks.append(None)
	side = []
	for k in range(2):
		if pred_masks[k] is not None and gt_masks[k] is not None:
			side.append(iou(pred_masks[k], gt_masks[k]))
		else:
			side.append(np.nan)
	P = [m for m in pred_masks if m is not None]
	G = [m for m in gt_masks if m is not None]
	matched = []
	if len(P) > 0 and len(G) > 0:
		M = np.zeros((len(P), len(G)))
		for a in range(len(P)):
			for b in range(len(G)):
				M[a, b] = iou(P[a], G[b])
		r, c = linear_sum_assignment(-M)
		matched = list(M[r, c])
	return {'n_pred': len(P), 'n_gt': len(G), 'matched': matched, 'side': side,
	        'pred_on': [m is not None for m in pred_masks], 'gt_on': [m is not None for m in gt_masks]}

def prf(tp, fp, fn):
	if tp + fp + fn == 0:
		return np.nan, np.nan, np.nan
	p = tp / (tp + fp) if tp + fp > 0 else 0.0
	r = tp / (tp + fn) if tp + fn > 0 else 0.0
	f1 = 2 * p * r / (p + r) if p + r > 0 else 0.0
	return p, r, f1

def frame_stats(records):
	rows = []
	for r in records:
		rows.append([sum(m > 0.5 for m in r['matched']), sum(m > 0.7 for m in r['matched']), r['n_pred'], r['n_gt'],
		             int(r['side'][0] > 0.5), int(r['side'][1] > 0.5), int(r['pred_on'][0]), int(r['pred_on'][1]),
		             int(r['gt_on'][0]), int(r['gt_on'][1])])
	return np.array(rows, dtype=np.int64)

def scores_from(stats):
	s = stats.sum(0)
	out = {'fp': int(s[2] - s[0])}
	if s[3] == 0:
		out.update({'precision': np.nan, 'recall': np.nan, 'f1': np.nan, 'f1@0.7': np.nan})
		return out
	p, r, f1 = prf(s[0], s[2] - s[0], s[3] - s[0])
	out.update({'precision': p, 'recall': r, 'f1': f1, 'f1@0.7': prf(s[1], s[2] - s[1], s[3] - s[1])[2]})
	return out

def bootstrap_f1(stats, clips, n=1000):
	ids, inverse = np.unique(clips, return_inverse=True)
	per_clip = np.zeros((len(ids), stats.shape[1]))
	np.add.at(per_clip, inverse, stats)
	rng = np.random.default_rng(SEED)
	values = []
	for _ in range(n):
		s = per_clip[rng.integers(0, len(ids), len(ids))].sum(0)
		values.append(prf(s[0], s[2] - s[0], s[3] - s[0])[2])
	return np.nanpercentile(values, [2.5, 97.5])

def evaluate(lanes, split):
	gt, meta = GT[split]
	with ThreadPoolExecutor(8) as pool:
		records = list(pool.map(frame_ious, lanes, gt, [SPLIT_DS[split]] * len(gt)))
	return frame_stats(records)

def smooth_lanes(lanes, sigma, degree):
	out = np.full_like(lanes, np.nan)
	for i in range(len(lanes)):
		for k in range(2):
			rows = np.where(np.isfinite(lanes[i, k]))[0]
			if len(rows) > degree + 2:
				weights = 1 / (np.nan_to_num(sigma[i, k, rows], nan=5.0) + 1)
				coef = np.polyfit(rows, lanes[i, k, rows], degree, w=weights)
				top, bottom = rows.min(), rows.max()
				out[i, k, top:bottom + 1] = np.polyval(coef, np.arange(top, bottom + 1))
	return out

def to_lanes(mean, spread, prob, threshold):
	xs = X_MIN + (mean + 0.5) * CELL
	sig = spread * CELL
	n = len(mean)
	lanes = np.full((n, 2, H), np.nan, np.float32)
	sigma = np.full((n, 2, H), np.nan, np.float32)
	conf = np.zeros((n, 2), np.float32)
	for i in range(n):
		for k in range(2):
			on = prob[i, k] > threshold
			if on.sum() >= MIN_ROWS:
				lanes[i, k] = resample(xs[i, k, on], ROWS[on])
				sigma[i, k] = resample(sig[i, k, on], ROWS[on])
				conf[i, k] = prob[i, k, on].mean()
	if SMOOTH is not None:
		lanes = smooth_lanes(lanes, sigma, SMOOTH)
	return lanes, conf, sigma

In [5]:
@torch.no_grad()
def embed_split(split, dino):
	path = f"{FUSION_DIR}/dino_cls_{split}.npy"
	if os.path.exists(path):
		return np.load(path)
	images = load(split)
	out = []
	for start in tqdm(range(0, len(images), 64), desc=split):
		x = (np.asarray(images[start:start + 64]) / 255.0 - MEAN) / STD
		x = torch.tensor(x.transpose(0, 3, 1, 2), dtype=torch.float32, device='cuda')
		with torch.autocast('cuda', dtype=torch.float16):
			cls = dino(pixel_values=x).last_hidden_state[:, 0]
		out.append(cls.float().cpu().numpy())
	emb = np.concatenate(out)
	np.save(path, emb)
	return emb

if all(os.path.exists(f"{FUSION_DIR}/dino_cls_{s}.npy") for s in SPLITS):
	EMB = {s: np.load(f"{FUSION_DIR}/dino_cls_{s}.npy") for s in SPLITS}
else:
	dino = AutoModel.from_pretrained(DINO).cuda().eval()
	EMB = {s: embed_split(s, dino) for s in SPLITS}
	del dino
pca = PCA(16, random_state=SEED).fit(EMB['val'])
EMB16 = {s: pca.transform(EMB[s]) for s in SPLITS}
print({s: EMB16[s].shape for s in SPLITS})

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 86.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

val:   0%|          | 0/149 [00:00<?, ?it/s]

test_culane:   0%|          | 0/60 [00:00<?, ?it/s]

test_bdd:   0%|          | 0/40 [00:00<?, ?it/s]

{'val': (9500, 16), 'test_culane': (3798, 16), 'test_bdd': (2560, 16)}


In [6]:
COLUMNS = ['source', 'side', 'conf', 'n_rows', 'top', 'bottom', 'x200', 'x120', 'sigma', 'agreement', 'other_source_present',
           'other_side_present', 'width200', 'base_prob_mean', 'base_prob_max'] + [f'emb{j}' for j in range(16)]

def value_at(lane, row):
	if np.isfinite(lane[row]):
		return float(lane[row])
	return np.nan

def agreement(a, b):
	both = np.isfinite(a) & np.isfinite(b)
	if both.sum() < 5:
		return np.nan
	return float(np.mean(np.abs(a[both] - b[both])))

def build_candidates(base_run, split):
	d = np.load(f"{CACHE_DIR}/{base_run}_{split}.npz")
	base_lanes, base_conf, base_sigma = to_lanes(d['mean'], d['spread'], d['prob'], 0.2)
	base_prob = d['prob']
	y = np.load(f"{FUSION_DIR}/yolo_{split}.npz")
	sources = [(base_lanes, base_conf, base_sigma), (y['lanes'], y['conf'], y['sigma'])]
	rows = []
	cands = []
	for i in range(len(base_lanes)):
		for k in range(2):
			for s in range(2):
				lanes, conf, sigma = sources[s]
				lane = lanes[i, k]
				r = np.where(np.isfinite(lane))[0]
				if len(r) < 2:
					continue
				other = sources[1 - s][0][i, k]
				sig = sigma[i, k, r]
				rows.append([s, k, conf[i, k], len(r), r.min(), r.max(), value_at(lane, 200), value_at(lane, 120),
				             float(np.nanmean(sig)) if np.isfinite(sig).any() else np.nan,
				             agreement(lane, other), float(np.isfinite(other).sum() > 1), float(np.isfinite(lanes[i, 1 - k]).sum() > 1),
				             value_at(lanes[i, 1], 200) - value_at(lanes[i, 0], 200), float(base_prob[i, k].mean()), float(base_prob[i, k].max())]
				            + list(EMB16[split][i]))
				cands.append((i, k, s))
	return pd.DataFrame(rows, columns=COLUMNS), np.array(cands), sources

def candidate_ious(base_run, cands, sources):
	path = f"{FUSION_DIR}/targets_{base_run}_val.npy"
	if os.path.exists(path):
		return np.load(path)
	gt = GT['val'][0]

	def one(c):
		i, k, s = c
		mask = lane_mask(sources[s][0][i, k], 'culane')
		best = 0.0
		for g in gt[i]:
			if np.isfinite(g).sum() > 1:
				best = max(best, iou(mask, lane_mask(g, 'culane')))
		return best

	with ThreadPoolExecutor(8) as pool:
		ious = np.array(list(tqdm(pool.map(one, cands), total=len(cands), desc='val targets')))
	np.save(path, ious)
	return ious

DATA = {}
for base_run in ['ensemble_final', 'r18_640x224_smooth']:
	for split in SPLITS:
		DATA[(base_run, split)] = build_candidates(base_run, split)
	X, cands, sources = DATA[(base_run, 'val')]
	DATA[(base_run, 'val_ious')] = candidate_ious(base_run, cands, sources)
	print(base_run, '| val candidates', len(cands), '| correct (IoU > 0.5):', round((DATA[(base_run, 'val_ious')] > 0.5).mean(), 3))

val targets:   0%|          | 0/35644 [00:00<?, ?it/s]

ensemble_final | val candidates 35644 | correct (IoU > 0.5): 0.728


val targets:   0%|          | 0/35848 [00:00<?, ?it/s]

r18_640x224_smooth | val candidates 35848 | correct (IoU > 0.5): 0.713


In [9]:
BASE_COLUMNS = COLUMNS[:15]
EMB_COLUMNS = COLUMNS[15:]
VARIANTS = [
	('rescore_ensemble', 'ensemble_final', False, False),
	('fusion_ensemble_yolo', 'ensemble_final', True, False),
	('fusion_ensemble_yolo_dino', 'ensemble_final', True, True),
	('fusion_r18_yolo', 'r18_640x224_smooth', True, False),
	('rescore_r18', 'r18_640x224_smooth', False, False),
]

def choose(cands, probs, threshold):
	best = {}
	for j in range(len(cands)):
		i, k, s = cands[j]
		if probs[j] > threshold:
			if (i, k) not in best or probs[j] > probs[best[(i, k)]]:
				best[(i, k)] = j
	return best

def make_output(best, cands, sources, probs, n):
	lanes = np.full((n, 2, H), np.nan, np.float32)
	sigma = np.full((n, 2, H), np.nan, np.float32)
	conf = np.zeros((n, 2), np.float32)
	for (i, k), j in best.items():
		s = cands[j][2]
		lanes[i, k] = sources[s][0][i, k]
		sigma[i, k] = sources[s][2][i, k]
		conf[i, k] = probs[j]
	return lanes, conf, sigma

val_gt_lanes = int((np.isfinite(GT['val'][0]).sum(2) > 1).sum())
val_clips = GT['val'][1].clip_id.values
done = pd.read_csv(RESULTS_FUSION_CSV) if os.path.exists(RESULTS_FUSION_CSV) else pd.DataFrame(columns=['run'])

for name, base_run, use_yolo, use_emb in VARIANTS:
	if name in done.run.values:
		print(name, 'already finished')
		continue
	columns = list(BASE_COLUMNS)
	if not use_yolo:
		columns = [c for c in columns if c not in ('source', 'agreement', 'other_source_present')]
	if use_emb:
		columns = columns + EMB_COLUMNS

	X, cands, sources = DATA[(base_run, 'val')]
	ious = DATA[(base_run, 'val_ious')]
	keep = np.ones(len(cands), bool) if use_yolo else cands[:, 2] == 0
	Xv, cv, iv = X.loc[keep, columns].reset_index(drop=True), cands[keep], ious[keep]
	y = (iv > 0.5).astype(int)
	groups = val_clips[cv[:, 0]]
	oof = np.zeros(len(y))
	for train_idx, test_idx in GroupKFold(5).split(Xv, y, groups):
		model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=SEED)
		model.fit(Xv.iloc[train_idx], y[train_idx])
		oof[test_idx] = model.predict_proba(Xv.iloc[test_idx])[:, 1]
	best_threshold, best_val_f1 = 0.5, -1
	for threshold in np.arange(0.1, 0.91, 0.05):
		chosen = choose(cv, oof, threshold)
		tp = sum(iv[j] > 0.5 for j in chosen.values())
		f1 = prf(tp, len(chosen) - tp, val_gt_lanes - tp)[2]
		if f1 > best_val_f1:
			best_threshold, best_val_f1 = float(threshold), f1
	model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=SEED)
	model.fit(Xv, y)

	row = {'run': name, 'base': base_run, 'yolo': use_yolo, 'dino_embedding': use_emb, 'threshold': best_threshold, 'val_f1_estimate': best_val_f1}
	for split, short in [('test_culane', 'culane'), ('test_bdd', 'bdd')]:
		X, cands, sources = DATA[(base_run, split)]
		keep = np.ones(len(cands), bool) if use_yolo else cands[:, 2] == 0
		probs = model.predict_proba(X.loc[keep, columns].reset_index(drop=True))[:, 1]
		n = len(GT[split][0])
		lanes, conf, sigma = make_output(choose(cands[keep], probs, best_threshold), cands[keep], sources, probs, n)
		os.makedirs(f"{PREDICTIONS_DIR}/{name}", exist_ok=True)
		np.savez_compressed(f"{PREDICTIONS_DIR}/{name}/{split}.npz", lanes=lanes, conf=conf, sigma=sigma)
		stats = evaluate(lanes, split)
		np.save(f"{STATS_DIR}/{name}_{split}.npy", stats)
		s = scores_from(stats)
		low, high = bootstrap_f1(stats, GT[split][1].clip_id.values)
		row.update({f'{short}_f1': s['f1'], f'{short}_ci_low': low, f'{short}_ci_high': high, f'{short}_precision': s['precision'],
		            f'{short}_recall': s['recall'], f'{short}_f1@0.7': s['f1@0.7'], f'{short}_fp': s['fp']})
	print(row)
	done = pd.concat([done, pd.DataFrame([row])], ignore_index=True)
	done.to_csv(RESULTS_FUSION_CSV, index=False)

rescore_ensemble already finished
fusion_ensemble_yolo already finished
fusion_ensemble_yolo_dino already finished
fusion_r18_yolo already finished
{'run': 'rescore_r18', 'base': 'r18_640x224_smooth', 'yolo': False, 'dino_embedding': False, 'threshold': 0.3500000000000001, 'val_f1_estimate': np.float64(0.7980080921257391), 'culane_f1': np.float64(0.6994955971616654), 'culane_ci_low': np.float64(0.6698412062063519), 'culane_ci_high': np.float64(0.7259065118124971), 'culane_precision': np.float64(0.8038907447435646), 'culane_recall': np.float64(0.6190980629539952), 'culane_f1@0.7': np.float64(0.5450970334273745), 'culane_fp': 998, 'bdd_f1': np.float64(0.11410233553218042), 'bdd_ci_low': np.float64(0.10147302928646879), 'bdd_ci_high': np.float64(0.1267374791069723), 'bdd_precision': np.float64(0.21290751829673984), 'bdd_recall': np.float64(0.07793472966390648), 'bdd_f1@0.7': np.float64(0.0071313959707612765), 'bdd_fp': 1183}


In [10]:
reference = pd.read_csv(f"{RESULTS_DIR}/results_test.csv").set_index('run')
fusion = pd.read_csv(RESULTS_FUSION_CSV).set_index('run')
rows = []
for run in ['ensemble_final', 'r18_640x224_smooth', 'yolo26n_seg']:
	r = reference.loc[run]
	rows.append({'run': run, 'culane_f1': r.culane_f1, 'culane_ci_low': r.culane_ci_low, 'culane_ci_high': r.culane_ci_high,
	             'culane_precision': r.culane_precision, 'culane_recall': r.culane_recall, 'bdd_f1': r.bdd_f1})
for run in fusion.index:
	r = fusion.loc[run]
	rows.append({'run': run, 'culane_f1': r.culane_f1, 'culane_ci_low': r.culane_ci_low, 'culane_ci_high': r.culane_ci_high,
	             'culane_precision': r.culane_precision, 'culane_recall': r.culane_recall, 'bdd_f1': r.bdd_f1})
display(pd.DataFrame(rows).round(4))

meta = GT['test_culane'][1]
table = {}
for run in ['ensemble_final', 'r18_640x224_smooth', 'yolo26n_seg'] + list(fusion.index):
	stats = np.load(f"{STATS_DIR}/{run}_test_culane.npy")
	row = {}
	for condition in sorted(meta.condition.unique()):
		s = scores_from(stats[meta.condition.values == condition])
		row[condition] = s['fp'] if condition == 'cross' else s['f1']
	table[run] = row
print('test_culane per condition (cross = false positives out of 422 frames)')
display(pd.DataFrame(table).T.round(3))

,run,culane_f1,culane_ci_low,culane_ci_high,culane_precision,culane_recall,bdd_f1
0,ensemble_final,0.6930,0.6578,0.7222,0.6827,0.7037,0.1686
1,r18_640x224_smooth,0.6559,0.6253,0.6828,0.6427,0.6696,0.1434
2,yolo26n_seg,0.6628,0.6311,0.6898,0.7779,0.5773,0.1606
3,rescore_ensemble,0.7294,0.6986,0.7554,0.8151,0.6600,0.1625
4,fusion_ensemble_yolo,0.7291,0.6998,0.7547,0.8125,0.6612,0.1662
5,fusion_ensemble_yolo_dino,0.7312,0.7022,0.7556,0.8033,0.6710,0.1661
6,fusion_r18_yolo,0.7084,0.6786,0.7343,0.8063,0.6317,0.1524
7,rescore_r18,0.6995,0.6698,0.7259,0.8039,0.6191,0.1141


test_culane per condition (cross = false positives out of 422 frames)


,arrow,cross,crowd,curve,hlight,night,noline,normal,shadow
ensemble_final,0.855,273.0,0.757,0.675,0.623,0.688,0.467,0.898,0.688
r18_640x224_smooth,0.823,383.0,0.712,0.627,0.604,0.660,0.437,0.876,0.648
yolo26n_seg,0.758,102.0,0.729,0.594,0.536,0.705,0.444,0.836,0.681
rescore_ensemble,0.862,133.0,0.790,0.701,0.613,0.744,0.519,0.902,0.713
fusion_ensemble_yolo,0.869,121.0,0.795,0.689,0.614,0.735,0.514,0.896,0.721
fusion_ensemble_yolo_dino,0.863,135.0,0.790,0.695,0.635,0.745,0.515,0.903,0.717
fusion_r18_yolo,0.842,134.0,0.763,0.643,0.611,0.737,0.491,0.876,0.709
rescore_r18,0.837,140.0,0.760,0.605,0.612,0.731,0.492,0.875,0.687


In [11]:
import joblib
MODELS_DIR = f"{BASE_DIR}/models"

def columns_for(use_yolo, use_emb):
	columns = list(BASE_COLUMNS)
	if not use_yolo:
		columns = [c for c in columns if c not in ('source', 'agreement', 'other_source_present')]
	if use_emb:
		columns = columns + EMB_COLUMNS
	return columns

done = pd.read_csv(RESULTS_FUSION_CSV).set_index('run')
for name, base_run, use_yolo, use_emb in VARIANTS:
	path = f"{MODELS_DIR}/{name}/meta.joblib"
	if os.path.exists(path):
		print(name, 'already saved')
		continue
	columns = columns_for(use_yolo, use_emb)
	X, cands, sources = DATA[(base_run, 'val')]
	ious = DATA[(base_run, 'val_ious')]
	keep = np.ones(len(cands), bool) if use_yolo else cands[:, 2] == 0
	model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=SEED)
	model.fit(X.loc[keep, columns].reset_index(drop=True), (ious[keep] > 0.5).astype(int))
	os.makedirs(f"{MODELS_DIR}/{name}", exist_ok=True)
	joblib.dump({'model': model, 'columns': columns, 'threshold': float(done.loc[name, 'threshold']), 'base_run': base_run,
	             'base_threshold': 0.2, 'smooth': SMOOTH, 'use_yolo': use_yolo, 'use_emb': use_emb,
	             'pca': pca if use_emb else None}, path)
	print(name, 'saved to', path)

rescore_ensemble saved to /content/drive/MyDrive/lane detection/models/rescore_ensemble/meta.joblib
fusion_ensemble_yolo saved to /content/drive/MyDrive/lane detection/models/fusion_ensemble_yolo/meta.joblib
fusion_ensemble_yolo_dino saved to /content/drive/MyDrive/lane detection/models/fusion_ensemble_yolo_dino/meta.joblib
fusion_r18_yolo saved to /content/drive/MyDrive/lane detection/models/fusion_r18_yolo/meta.joblib
rescore_r18 saved to /content/drive/MyDrive/lane detection/models/rescore_r18/meta.joblib
